

**SOPHY 把数据分成两条链处理：**
一条是 **geometry 链**，负责生成/增强 3D mesh 和纹理；
另一条是 **dynamics 链**，负责把 mesh 变成可仿真的粒子，并按相机视角渲染成视频。

---

## 1. 它的数据目录是怎么分工的

仓库默认假设数据长这样：

* `data/...`：相机、视角、渲染图像
* `simulation_data/...`：mesh、mtl、贴图、采样点、材料参数

README 里也明确写了：

* `data/` 保存原始 3D 对象的渲染图
* `simulation_data/` 保存 mesh 和 material parameters

所以对一个对象来说，通常是：

* `data/train/<category>/<obj_id>/...`
* `simulation_data/train/<category>/<obj_id>/...`

---

## 2. 相机数据怎么读

这部分在 `dynamics/util/dataset.py` 的 `ViewDataset`。

它做的事是：

1. 读 `transforms.json` 或 `data_static.json`
2. 逐项取出：

   * `c2w`
   * `intrinsic`
   * `file_path`
3. 把 `c2w` 求逆得到 `w2c`
4. 用内参里的焦距算 FOV
5. 用 Kaolin 建相机对象

所以它真正依赖的是：

* 每个视角/帧的 `c2w`
* 每个视角/帧的 `intrinsic`

最后 `dataset[i]` 返回的不是图像，而是 **camera**。
图像本身在 dynamics 推理里不是直接读进来做网络输入，而是主要拿相机来渲染仿真 mesh。

---

## 3. mesh 是怎么变成“可仿真数据”的

这部分核心在 `dynamics/util/dataprep.py` 的 `prepare_simulation_data_given_mesh()`。

它做了几件关键事：

### 3.1 读 mesh

只支持 `.obj`，然后：

* `trimesh.load_mesh(mesh_path, force='mesh', process=False)`
* `kal.io.obj.import_mesh(..., raw_materials=True)`

这里分成两套表示：

* `trimesh`：用于几何处理、体采样
* `kaolin mesh`：用于后续渲染

### 3.2 如果 mesh 不是 watertight，就先修

如果网格不闭合，而且没提供现成粒子点云：

* 调 `ManifoldPlus`
* 生成 `*.mp.obj`
* 再重新读

### 3.3 从 mesh 采样粒子

支持三种模式：

* `uniform`
* `volumetric`
* `surface`

其中最重要的是 `volumetric_sampling()`，它会在物体体积内部采点，供 MPM 用。

### 3.4 输出四样东西

它最终返回：

* `mesh_vertices`
* `particles`
* `vol_all`
* `kal_mesh`

也就是：

* 网格顶点
* 体粒子
* 体积
* 带材质信息的渲染 mesh

---

## 4. 物理仿真时，数据怎么组织

这部分在 `dynamics/mpm/object_utils.py` 里，核心类是 `PhysObject`。

### 4.1 一个对象先变成两部分点

它会把：

* `vertices`
* `particles`

拼起来：

```python
comb_pos = np.concatenate([vertices, particles], axis=0)
```

也就是说，**mesh 顶点和仿真粒子会放在同一个大点集里**。

### 4.2 为什么要这样拼

因为后面仿真更新的是整套点的位置，而渲染时只需要取出“mesh 顶点对应那一部分点”。

所以它专门构造了一个 `mv_indices`：

* 前 `vertices.shape[0]` 个位置标 1
* 后面的粒子标 0

后面每一步仿真结束后，都用这个 mask 从全部粒子位置里把 mesh 顶点抽出来，更新渲染 mesh。

这一步非常关键：
**仿真驱动的是粒子，渲染显示的是 mesh，但 mesh 顶点位置来自粒子状态更新。**

---

## 5. 材料参数是怎么进入仿真的

这部分也在 `dynamics/mpm/object_utils.py`，主要是 `prepare_materials()`。

它分三种情况：

### 5.1 generated object

如果 `material.generated = true`，它会去读：

* `sampled_points_info.npz`

然后按粒子级别恢复：

* `elasticity`
* `plasticity`
* `E`
* `nu`
* `sigma`
* `phi`

### 5.2 dataset object 且有 part-level 材料信息

如果有 `part_labels` 和 `mat_labels`，再配合 `part_mat_params` 或某个 JSON，就会按 part 分配材料参数。

这里调用的是 `dynamics/mpm/compat_utils.py` 里的：

* `prepare_material_params_given_values()`
* 或 `prepare_material_params_given_npz()`

### 5.3 没有细分标签

就直接用 config 里给的整物体材料参数。

---

## 6. 你前面提到的 `mat_params_new_v3.4.json`，在这套代码里属于哪一层

结论很明确：

**它主要是给仿真用的，不是给渲染用的。**

因为 `prepare_material_params_given_values()` 读的是这种 part-level 信息：

* `E`
* `nu`
* `sigma_y`
* `elasticity`
* `plasticity`
* `mmid`
* `mat_name`
* `mat_sub_type`

然后把这些映射成 **每个仿真点的材料参数**。

所以像你给的：

* `bag_clip_buckle -> metal`
* `bag_body -> fabric`
* `handle -> plastic`

在当前代码里最直接的用途是：

* buckle 更硬、更金属化的力学行为
* bag body 更软、更易形变
* handle/connector 介于两者之间

**但它并不会直接参与贴图渲染。**

---

## 7. 纹理和外观是怎么渲染出来的

这部分在 `dynamics/util/dataprep.py` 的 `mesh_rasterization()`。

这段非常关键，因为它直接回答了你之前的问题：

### 它确实会读 OBJ/MTL 里的贴图

渲染时它做了：

1. 用 `kal.io.obj.import_mesh(..., raw_materials=True)` 读 mesh
2. 拿到：

   * `mesh.uvs`
   * `mesh.face_uvs_idx`
   * `mesh.materials`
   * `mesh.material_assignments`
3. 对每个 material：

   * 如果有 `map_Kd`，就把贴图拿出来
   * 否则退化成纯 `Kd` 颜色
4. 用 `dr.texture(...)` 按 UV 采样贴图

所以这条链里，**真正控制外观的是 OBJ/MTL/PNG**，不是材料 JSON。

也就是说：

* `material.obj`
* `material.mtl`
* `material_0~3.png`

这套资产，如果读通了，是会直接影响最终渲染外观的。

---

## 8. 生成模型出来的 mesh，怎么接到 dynamics 里

这部分是 geometry 链到 dynamics 链的桥，在两个文件里：

### `geometry/texture_enhance.py`

它对生成出来的 `.ply` mesh 做：

* 纹理增强
* 导出 `.obj + .mtl + .png`

而且它会把 Hunyuan3D 生成的默认 `material_0.png` 等名字，改成和 mesh 同名的单纹理形式。

### `geometry/link_generated_data.py`

它负责把生成结果整理成 dynamics 能吃的目录结构：

* 在 `generated_cache/.../simulation_data/...` 下软链接：

  * `mat_params_new_v3.4.json`
  * `sampled_points.ply`
  * `sampled_points_info.npz`
  * `obj`
  * `mtl`
  * `png`
* 在 `generated_cache/.../data/...` 下复制：

  * `data_static.json`
  * `cond.png` 或 `cond.txt`

所以这一步相当于把 geometry 产物“伪装成”标准数据集格式，让 dynamics 后处理直接复用。

---

## 9. inference 时完整流程是什么

`dynamics/inference.py` 的主流程是：

1. 读 config
2. 建 `ViewDataset`，拿所有 camera
3. 对每个 object 建 `PhysObject`
4. `prepare_simulation_environment_given_mesh()`

   * 合并所有对象的粒子
   * 建 MPM state/model/solver
   * 写入材料参数
5. 如果有边界条件，调用 `prepare_boundary_conditions_given_mesh()`
6. 开始时间步推进
7. 每隔 `skip_frames`：

   * 更新 mesh 顶点
   * 从指定 camera 视角 rasterize
   * 保存 PNG
8. 最后把 PNG 打包成 gif/mp4

---

## 10. 对你现在最相关的几个结论

### 结论 1：JSON 和贴图是两条并行信息

* `mat_params_new_v3.4.json`：主要进仿真
* `obj/mtl/png`：主要进渲染

### 结论 2：SOPHY 的渲染链确实支持多材质贴图

`mesh_rasterization()` 是按 `material_assignments + UV + map_Kd` 逐材质采样的，不是简单单色渲染。

### 结论 3：它的数据处理中，mesh 顶点和体粒子是绑定更新的

不是“单独模拟 mesh”，而是：

* 用粒子做 MPM
* 用粒子位置回写 mesh 顶点
* 再渲染 mesh

### 结论 4：如果你想把自己的 bag 数据接进来，最关键的是两点

一是 `simulation_data/...` 里这套资产要齐：

* `material.obj`
* `material.mtl`
* `material_*.png`
* `sampled_points.ply`
* `sampled_points_info.npz`
* `mat_params_new_v3.4.json`

二是 `data/...` 里要有：

* `transforms.json` 或 `data_static.json`

---

## 11. 这里还有一个你要特别注意的小坑

你这个 bag 是：

* `material.mtl`
* `material_0.png ~ material_3.png`

而 `prepare_simulation_data_given_mesh()` 里有一段简化假设：

```python
safe_symlink(mesh_path.with_suffix('.png'), ...)
```

这更像是按“单张同名贴图”写的，不是按 `material_0~3.png` 写的。

不过它后面真正导入 Kaolin mesh 时用的是原始 `mesh_path`，不是刚 symlink 出来的路径，所以**只要原始 OBJ/MTL 相对路径是正确的，多材质贴图大概率仍然能被正常读到**。
只是这说明：这段辅助缓存代码对“多张贴图”的兼容写得不够完整。

---

## 一句话总结

**SOPHY 处理数据的核心逻辑是：**
先用 `simulation_data` 里的 mesh 和材料参数，把对象变成“mesh 顶点 + 体粒子 + 每粒子材料属性”的 MPM 对象；
再用 `data` 里的相机参数，把仿真后的 deformable mesh 从多视角渲染成视频。
其中 **JSON 决定物理属性，OBJ/MTL/PNG 决定渲染外观**。

你要的话，我下一条可以继续帮你做两件事里的任意一个：
一是专门顺着 `bag/01_000__0` 这组文件，帮你画出“这个对象从读盘到渲染”的具体路径；二是直接指出你现在要改哪里，才能让 Genesis 也尽量复用这套外观和材料信息。
